# Stage 15 — Soumission : entraînement sur fenêtres consécutives 4-6 mois

**L'idée :** l'énoncé précise que chaque point de test a une **fenêtre consécutive de 4-6 mois** (le reste masqué), radar préservé, optique avec trous de nuages. On entraîne le modèle avec une corruption qui imite *exactement* ça, pour qu'il apprenne le bon scénario.

**Honnêteté :** en CV ce modèle ne montre pas un gain net (notre CV reste optimiste car elle ne capture pas le décalage train/test entre régions). On soumet pour voir le comportement réel sur le test. Attendu : autour de 0.82, mais ça vaut la vérification car le scénario d'entraînement est plus fidèle qu'avant.

In [3]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"; TEST_PATH="Test.csv"; SAMPLE_PATH="SampleSubmission.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
OPTICAL=["blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
def sr(a,b): return (a-b)/(a+b)

def build_features(df):
    feats={}; idx={}
    for m in MONTHS:
        g,n,r,s1=df[f"green_{m}"],df[f"nir_{m}"],df[f"red_{m}"],df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(sr(g,n)); idx.setdefault("MNDWI",[]).append(sr(g,s1))
        idx.setdefault("NDVI",[]).append(sr(n,r)); idx.setdefault("NDMI",[]).append(sr(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def s(mat,nm):
        a=np.column_stack([np.asarray(x,float) for x in mat])
        feats[f"{nm}_mean"]=np.nanmean(a,1);feats[f"{nm}_std"]=np.nanstd(a,1);feats[f"{nm}_min"]=np.nanmin(a,1)
        feats[f"{nm}_max"]=np.nanmax(a,1);feats[f"{nm}_range"]=np.nanmax(a,1)-np.nanmin(a,1);feats[f"{nm}_med"]=np.nanmedian(a,1)
        feats[f"{nm}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: s([df[f"{b}_{m}"] for m in MONTHS],b)
    for k,v in idx.items(): s(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)

def corrupt_window(df, seed=0):
    """Fenetre consecutive 4-6 mois (le reste masqué), radar preservé, optique avec trous de nuages."""
    rng=np.random.RandomState(seed); d=df.copy()
    cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        win=rng.choice([4,5,6]); start=rng.randint(0,12-win+1)
        active=np.zeros(12,bool); active[start:start+win]=True
        for b in BANDS: arr[b][i,~active]=np.nan               # hors fenetre: tout masqué
        cloud=active & (rng.rand(12)<0.3)                       # nuages dans la fenetre
        for b in OPTICAL: arr[b][i,cloud]=np.nan                # optique seulement
    for b in BANDS: d[cols[b]]=arr[b]
    return d

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

train=pd.read_csv(TRAIN_PATH).replace(NODATA,np.nan)
test=pd.read_csv(TEST_PATH).replace(NODATA,np.nan)
sample=pd.read_csv(SAMPLE_PATH)
y=train["label"].values

# augmentation avec fenetres consecutives (3 copies pour plus de variété de fenetres)
parts=[train]+[corrupt_window(train,seed=i+1) for i in range(3)]
aug=pd.concat(parts,ignore_index=True); yaug=np.concatenate([y]*len(parts))
X_aug=build_features(aug); X_test=build_features(test)

hgb=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=7,
        l2_regularization=1.0,random_state=42).fit(X_aug,yaug)
lr=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
        LogisticRegression(max_iter=2000)).fit(X_aug,yaug)
proba=0.5*hgb.predict_proba(X_test)[:,1]+0.5*lr.predict_proba(X_test)[:,1]

sub=pd.DataFrame({"ID":test["ID"],"TargetF1":(proba>=0.5).astype(int),"TargetRAUC":proba})
assert list(sub.columns)==list(sample.columns) and set(sub.ID)==set(sample.ID) and len(sub)==len(sample)
print("bassins predits: %d (%.1f%%) | vrai taux ~54%%"%(sub.TargetF1.sum(),100*sub.TargetF1.mean()))
sub.to_csv("submission_window.csv",index=False)
print("ecrit: submission_window.csv")
sub.head()

bassins predits: 602 (58.4%) | vrai taux ~54%
ecrit: submission_window.csv


,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,1,0.966380
1,ID_TS_NEW_7SPRN3PB,1,0.993387
2,ID_TS_NEW_LZOWPHLC,1,0.684704
3,ID_TS_NEW_DN6TUF64,0,0.417096
4,ID_TS_NEW_95N82M49,0,0.001199


## À retenir
- Entraînement fidèle au scénario réel (fenêtres 4-6 mois consécutives).
- Si le board reste ~0.82, ça confirme que le mur n'est pas la forme des trous mais le **décalage de distribution entre régions** — et on raisonnera sur des features plus invariantes.
- **Note :** les chemins de fichiers sont `Train.csv`/`Test.csv` (renommés).